# 365 Probabilidades - Dia #168
## Qual a probabilidade de falar demais de moral afastar quem te lê?

**Tipo:** Comportamental · Redes sociais (análise própria em dados públicos)  
**Data de publicação:** 2026-11-28  
**Ferramenta:** Python  
**Decisão analisada:** Vale a pena carregar na indignação para ser lido?  
**Hashtag:** #365Probabilidades #Dia168

---

### 📖 A História

Esses dias, uma pessoa que eu seguia nas redes sociais fez uma sequência de posts indignada com um caso. Comecei a reparar e a lembrar dos últimos posts desse tipo que eu tinha visto, e o roteiro era quase sempre o mesmo… Começa com "é inadmissível", segue com "uma vergonha", "um absurdo" e "quem apoia isso é cúmplice", e termina pedindo que você compartilhe, se tiver coragem.

Às vezes a gente para e lê. Muitas vezes passa reto.

A ideia de que indignação dá alcance virou quase uma regra nas redes. Fui ver se existe um ponto em que a moral deixa de atrair e passa a afastar.

---

### 📚 O Conceito

Falar de certo e errado chama atenção. Brady e colegas mostraram em 2017 que cada palavra moral e emocional a mais num post sobre temas polêmicos se associava a mais compartilhamentos. O efeito foi replicado em 2025, menor do que o original, mas firme.

Candia, Atari, Kteily e Uzzi (2026) separaram duas coisas. A **carga moral** é o quanto um post, no todo, trata de moral. A **densidade moral** é o quanto as palavras de moral se concentram no texto. Um post pode tratar de um tema moral e ainda assim ter poucas palavras de julgamento. Outro pode ser quase só julgamento.

A hipótese deles é que a carga atrai e a concentração cansa. Na média dos 13 temas que analisaram, o engajamento foi maior numa densidade perto de 0,30, e caiu quando a moral se concentrava mais do que isso.

---

### 🧮 O Modelo

A análise é própria, feita nos dados que os autores publicaram no OSF (osf.io/wzymh). São 1.621.147 posts, com a densidade moral, a carga moral e o engajamento de cada um. Não há textos nem identificadores.

1. **Quanto se fala de moral.** A distribuição da densidade moral nas três plataformas.
2. **A curva.** O engajamento de cada faixa de densidade, em relação à média de posts do mesmo tema e com a mesma carga moral.
3. **As comparações (modelo central).** Cada post com densidade acima de 0,30 ("moral demais") é comparado com todos os posts com densidade entre 0,10 e 0,30 ("equilíbrio") do mesmo tema e do mesmo quinto de carga moral. A pergunta é em quantas comparações o post "moral demais" tem menos respostas, com empate valendo meio. Os 55 grupos são combinados com peso pelo número de posts "moral demais", e o intervalo de 95% vem do erro de cada grupo (Hanley e McNeil).
4. **A chance de se destacar.** A probabilidade de o post ficar entre os 10% mais respondidos do seu grupo, para quem fala demais e para quem fica no equilíbrio, com 20 mil sorteios Beta por grupo (semente 42).
5. **Os 13 pontos ideais.** O ponto de densidade de maior engajamento em cada tema, calculado pelos autores, com o intervalo de 95%.

**Fontes:**
- Candia, C., Atari, M., Kteily, N. & Uzzi, B. (2026). Saturation of moral language predicts lower content engagement on social media. *Nature Human Behaviour*. **1.621.147 posts** sobre 13 temas sociopolíticos (530.104 no Twitter, 1.048.653 no Reddit, 42.390 no 8chan). Dados e código em osf.io/wzymh.
- Brady, W. J., Rathje, S., Globig, L. K. & Van Bavel, J. J. (2025). Estimating the effect size of moral contagion in online networks: A pre-registered replication and meta-analysis. *PNAS Nexus*, 4(11). Replicação com **849.266** posts e meta-análise de 27 estudos de 5 laboratórios (**4.821.006** posts).
- Brady, W. J., Wills, J. A., Jost, J. T., Tucker, J. A. & Van Bavel, J. J. (2017). Emotion shapes the diffusion of moralized content in social networks. *PNAS*, 114(28), 7313-7318. N = 563.312.

**Nota metodológica sobre o fator ×0,80.** Não se aplica. Os dados são publicações e contagens de respostas, e não autorrelato.

**Como rodar:** os três arquivos `*_channel_full3.csv` precisam estar em `dados168/data/feature_files/`, ao lado deste notebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import rankdata

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

SEMENTE = 42
N_SIM = 20_000
rng = np.random.default_rng(SEMENTE)

VERMELHO = '#c0392b'
DOURADO = '#c8a84b'
VERDE = '#1a5f5a'
CINZA = '#7f8c8d'
CORES_PLAT = {'Twitter': VERDE, 'Reddit': DOURADO, '8 Chan': VERMELHO}

fmt = lambda x, c=1: f"{x:.{c}f}".replace('.', ',')
mil = lambda x: f"{x:,}".replace(',', '.')

print("✅ Bibliotecas carregadas")

In [ ]:
# --- DADOS ---

# Camada própria: dados publicados por Candia et al. (2026) no OSF (osf.io/wzymh)
PASTA = 'dados168/data/feature_files/'
COLS = ['platform', 'channel', 'total_n_replies', 'soft_moral_density_w2v', 'moral_w2v_cle']
df = pd.concat([pd.read_csv(PASTA + f + '_channel_full3.csv', usecols=COLS)
                for f in ['tweets', 'reddit', 'chan']]).dropna()
df = df.rename(columns={'soft_moral_density_w2v': 'densidade', 'moral_w2v_cle': 'carga',
                        'total_n_replies': 'respostas'})

# Os 13 pontos ideais calculados pelos autores (código 10_Main_DDR_models do OSF): ponto e erro padrão
otimos = pd.DataFrame([
    ('Twitter', 'Mudança climática', 0.5368, 0.0887), ('Twitter', 'Controle de armas', 0.1753, 0.0385),
    ('Twitter', 'Casamento gay', 0.2762, 0.0273), ('Twitter', 'Relatório Mueller', 0.3354, 0.0217),
    ('Twitter', 'MeToo', 0.2332, 0.0093), ('Twitter', 'Pós-Brexit', 0.1736, 0.0317),
    ('Reddit', 'Comunismo', 0.3397, 0.0168), ('Reddit', 'Conservador', 0.4323, 0.0749),
    ('Reddit', 'The_Donald', 0.4843, 0.0176), ('Reddit', 'Liberal', 0.2944, 0.0263),
    ('8 Chan', 'Leftypol', 0.4352, 0.0477), ('8 Chan', 'POL', 0.2906, 0.0179),
    ('8 Chan', 'TV', 0.4613, 0.0328)], columns=['plataforma', 'tema', 'otimo', 'ep'])
otimo_medio, ep_medio = 0.3006, 0.0713          # efeito aleatório dos 13 temas

# Segundo estudo: Brady et al. (2025), PNAS Nexus. Taxa de compartilhamento por palavra moral-emocional a mais
brady = dict(irr=1.13, ic=(1.06, 1.20), n_meta=4_821_006, n_rep=849_266, estudos=27, labs=5,
             irr_2017=1.20, n_2017=563_312)

aplica_fator_080 = False

print("=" * 72)
print("  DADOS - A MORAL QUE CANSA")
print("=" * 72)
print(f"\n  Candia et al. (2026), OSF: {mil(len(df))} posts")
for p, n in df.groupby('platform').size().items():
    print(f"  → {p:<8} {mil(n)} posts  ·  {df.loc[df.platform == p, 'channel'].nunique()} temas")
print(f"  → Ponto ideal médio dos 13 temas: {fmt(otimo_medio, 2)} (erro padrão {fmt(ep_medio, 3)})")
print(f"\n  Brady et al. (2025): {mil(brady['n_meta'])} posts em {brady['estudos']} estudos de {brady['labs']} laboratórios")
print(f"  → Cada palavra moral-emocional a mais: ×{fmt(brady['irr'], 2)} [{fmt(brady['ic'][0], 2)}; {fmt(brady['ic'][1], 2)}] nos compartilhamentos")
print(f"  → No estudo original de 2017 (N = {mil(brady['n_2017'])}): ×{fmt(brady['irr_2017'], 2)}")
print(f"\n  Fator ×0,80 aplicado: {'sim' if aplica_fator_080 else 'não'}")
print("=" * 72)

In [ ]:
# --- O MODELO ---

# Grupos de comparação: mesmo tema e mesmo quinto de carga moral
df['quinto_carga'] = df.groupby('channel')['carga'].transform(
    lambda s: pd.qcut(s.rank(method='first'), 5, labels=False))
df['faixa'] = np.select([df.densidade < 0.10, df.densidade < 0.30], ['pouca', 'equilibrio'], 'demais')

# Camada 1 · Quanto se fala de moral
p_acima_030 = (df.densidade >= 0.30).mean()
p_acima_050 = (df.densidade >= 0.50).mean()
p_acima_030_plat = (df.densidade >= 0.30).groupby(df.platform).mean()

# Camada 2 · A curva: engajamento relativo à média do grupo
FAIXAS = [0, .1, .2, .3, .4, .5, 1.01]
ROT_FAIXAS = ['0 a 0,1', '0,1 a 0,2', '0,2 a 0,3', '0,3 a 0,4', '0,4 a 0,5', '0,5 ou mais']
df['banda'] = pd.cut(df.densidade, FAIXAS, right=False, labels=ROT_FAIXAS)
df['relativo'] = df.respostas / df.groupby(['channel', 'quinto_carga']).respostas.transform('mean')
curva = df.groupby(['platform', 'banda'], observed=True).relativo.mean().unstack(0)

# Camada 3 · As comparações (probabilidade de superioridade, Mann-Whitney por grupo)
df['p90'] = df.groupby(['channel', 'quinto_carga']).respostas.transform(lambda s: s.quantile(0.9))
df['espalhou'] = (df.respostas > df.p90).astype(int)

def duelo(a, b):
    # P(a < b) + 0,5 · P(a = b), com o erro de Hanley e McNeil
    r = rankdata(np.concatenate([a, b]))
    U = r[:len(a)].sum() - len(a) * (len(a) + 1) / 2
    A = 1 - U / (len(a) * len(b))
    q1, q2 = A / (2 - A), 2 * A**2 / (1 + A)
    var = (A * (1 - A) + (len(a) - 1) * (q1 - A**2) + (len(b) - 1) * (q2 - A**2)) / (len(a) * len(b))
    return A, var

linhas = []
for (ch, q), g in df.groupby(['channel', 'quinto_carga']):
    a = g.respostas[g.faixa == 'demais'].values
    b = g.respostas[g.faixa == 'equilibrio'].values
    if len(a) >= 20 and len(b) >= 20:
        A, var = duelo(a, b)
        ea = g.espalhou[g.faixa == 'demais']; eb = g.espalhou[g.faixa == 'equilibrio']
        linhas.append(dict(platform=g.platform.iloc[0], tema=ch, n_demais=len(a), n_equil=len(b), A=A, var=var,
                           k_d=ea.sum(), k_e=eb.sum()))
grupos = pd.DataFrame(linhas)

def combina(t):
    w = t.n_demais / t.n_demais.sum()
    A = (w * t.A).sum(); ep = np.sqrt((w**2 * t['var']).sum())
    return pd.Series({'duelos': A * 100, 'ic_lo': (A - 1.96 * ep) * 100, 'ic_hi': (A + 1.96 * ep) * 100,
                      'n_demais': t.n_demais.sum(), 'grupos': len(t)})
duelos_plat = grupos.groupby('platform').apply(combina)
duelos_geral = combina(grupos)

# Camada 4 · A chance de se espalhar (padronizada pela composição dos posts "moral demais")
def chance(t, k, n):
    w = (t.n_demais / t.n_demais.sum()).values
    sims = rng.beta(t[k].values[:, None] + 0.5, (t[n] - t[k]).values[:, None] + 0.5, (len(t), N_SIM))
    s = (w[:, None] * sims).sum(0)
    return np.percentile(s * 100, [2.5, 50, 97.5])
espalha = {}
for nome, t in list(grupos.groupby('platform')) + [('Geral', grupos)]:
    espalha[nome] = dict(demais=chance(t, 'k_d', 'n_demais'), equil=chance(t, 'k_e', 'n_equil'))
queda_rel = 1 - espalha['Geral']['demais'][1] / espalha['Geral']['equil'][1]

# Camada 5 · Os 13 pontos ideais
ic_otimo = (otimo_medio - 1.96 * ep_medio, otimo_medio + 1.96 * ep_medio)

print("=" * 72)
print("  MODELO - RESULTADO")
print("=" * 72)
print(f"\n  CAMADA 1 · Posts acima de 0,30 de densidade: {fmt(p_acima_030*100)}%  ·  acima de 0,50: {fmt(p_acima_050*100)}%")
for p, v in p_acima_030_plat.items():
    print(f"  → {p:<8} {fmt(v*100)}%")
print("\n  CAMADA 2 · Engajamento relativo por faixa de densidade (1 = média do grupo)")
print(curva.round(2).to_string())
print("\n  CAMADA 3 · Em 100 comparações, em quantas o post 'moral demais' perde para o 'equilíbrio'")
for p, r in duelos_plat.iterrows():
    print(f"  → {p:<8} {fmt(r.duelos)} [{fmt(r.ic_lo)}; {fmt(r.ic_hi)}]  ·  {mil(int(r.n_demais))} posts em {int(r.grupos)} grupos")
print(f"  → GERAL    {fmt(duelos_geral.duelos)} [{fmt(duelos_geral.ic_lo)}; {fmt(duelos_geral.ic_hi)}]")
print("\n  CAMADA 4 · Chance de ficar entre os 10% mais respondidos do grupo")
for p, e in espalha.items():
    print(f"  → {p:<8} moral demais {fmt(e['demais'][1])}% [{fmt(e['demais'][0])}; {fmt(e['demais'][2])}]"
          f"  ·  equilíbrio {fmt(e['equil'][1])}% [{fmt(e['equil'][0])}; {fmt(e['equil'][2])}]")
print(f"  → Queda relativa na chance de se espalhar: {fmt(queda_rel*100, 0)}%")
print(f"\n  CAMADA 5 · Ponto ideal médio {fmt(otimo_medio, 2)} [IC 95% {fmt(ic_otimo[0], 2)}; {fmt(ic_otimo[1], 2)}]"
      f"  ·  temas de {fmt(otimos.otimo.min(), 2)} a {fmt(otimos.otimo.max(), 2)}")
print("=" * 72)

In [ ]:
# --- VISUALIZAÇÃO ---
RODAPE = 'Fonte: Candia, Atari, Kteily & Uzzi (2026), Nature Human Behaviour, dados do OSF. Análise própria | #365Probabilidades'
PLATS = ['Twitter', 'Reddit', '8 Chan']

# GRÁFICO 1 · Quanto se fala de moral
fig1, ax1 = plt.subplots(figsize=(12, 8))
dist = df.groupby(['platform', 'banda'], observed=True).size().unstack(0)
dist = dist / dist.sum() * 100
x = np.arange(len(ROT_FAIXAS)); larg = 0.27
for i, p in enumerate(PLATS):
    ax1.bar(x + (i - 1) * larg, dist[p].reindex(ROT_FAIXAS).values, width=larg, color=CORES_PLAT[p], label=p)
ax1.axvline(2.5, color=CINZA, ls='--', lw=1.5)
ax1.text(2.6, 50, f'só {fmt(p_acima_030*100)}% dos posts\npassam de 0,30', fontsize=13, color=VERMELHO, fontweight='bold')
ax1.set_xticks(x); ax1.set_xticklabels(ROT_FAIXAS, fontsize=12)
ax1.set_xlabel('Densidade moral (quanto as palavras de moral se concentram no post)', fontsize=12)
ax1.set_ylabel('Posts em cada faixa (%)'); ax1.legend(frameon=False, fontsize=12)
ax1.set_title(f'Quase ninguém fala só de moral\n{mil(len(df))} posts sobre 13 temas polêmicos', fontsize=14, pad=15)
plt.figtext(0.5, 0.005, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig('dia-168-grafico-01-quanto-se-fala.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 1 salvo!")

# GRÁFICO 2 · A curva
fig2, ax2 = plt.subplots(figsize=(12, 8))
for p in PLATS:
    ax2.plot(ROT_FAIXAS, curva[p].reindex(ROT_FAIXAS).values, marker='o', lw=2.5, ms=9, color=CORES_PLAT[p], label=p)
ax2.axhline(1, color=CINZA, ls='--', lw=1.2); ax2.text(5.05, 1.02, 'média do grupo', fontsize=10, color=CINZA)
ax2.set_ylim(0, 1.3); ax2.tick_params(axis='x', labelsize=12)
ax2.set_xlabel('Densidade moral', fontsize=12)
ax2.set_ylabel('Respostas em relação a posts do mesmo tema\ne com a mesma carga moral (1 = média)', fontsize=12)
ax2.legend(frameon=False, fontsize=12)
ax2.set_title('Quanto mais a moral se concentra, menos respostas\nEngajamento relativo por faixa de densidade', fontsize=14, pad=15)
plt.figtext(0.5, 0.005, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig('dia-168-grafico-02-a-curva.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 2 salvo!")

# GRÁFICO 3 · As comparações
fig3, ax3 = plt.subplots(figsize=(12, 8))
linhas3 = [(p, duelos_plat.loc[p], CORES_PLAT[p]) for p in PLATS] + [('Geral', duelos_geral, '#222222')]
ys = np.arange(len(linhas3))[::-1]
for y, (nome, r, cor) in zip(ys, linhas3):
    ax3.errorbar(r.duelos, y, xerr=[[r.duelos - r.ic_lo], [r.ic_hi - r.duelos]], fmt='D' if nome == 'Geral' else 'o',
                 color=cor, ms=13, capsize=6, lw=2.5)
    ax3.text(r.ic_hi + 1, y, f"{fmt(r.duelos, 0)} em 100", va='center', fontsize=15, fontweight='bold', color=cor)
ax3.axvline(50, color=DOURADO, ls='--', lw=2); ax3.text(50.3, -0.6, 'empate (50)', fontsize=11, color=DOURADO, fontweight='bold')
ax3.set_yticks(ys); ax3.set_yticklabels([l[0] for l in linhas3], fontsize=13)
ax3.set_xlim(40, 85); ax3.set_ylim(-0.9, len(linhas3) - 0.4)
ax3.set_xlabel('Em 100 comparações com um post equilibrado do mesmo tema e mesma carga moral,\nem quantas o post "moral demais" tem menos respostas', fontsize=12)
ax3.set_title('Falar demais de moral perde mais comparações do que ganha\nProbabilidade de superioridade com IC 95%', fontsize=14, pad=15)
plt.figtext(0.5, 0.005, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig('dia-168-grafico-03-as-comparacoes.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 3 salvo!")

# GRÁFICO 4 · A chance de se espalhar
fig4, ax4 = plt.subplots(figsize=(12, 8))
nomes = PLATS + ['Geral']; x = np.arange(len(nomes)); larg = 0.36
for j, (chave, cor, rot) in enumerate([('equil', VERDE, 'Equilíbrio (0,10 a 0,30)'), ('demais', VERMELHO, 'Moral demais (0,30 ou mais)')]):
    v = [espalha[n][chave] for n in nomes]
    meio = [e[1] for e in v]
    ax4.bar(x + (j - 0.5) * larg, meio, width=larg, color=cor, alpha=0.88, label=rot,
            yerr=[[e[1] - e[0] for e in v], [e[2] - e[1] for e in v]], capsize=6)
    for xi, m in zip(x, meio):
        ax4.text(xi + (j - 0.5) * larg, m + 1.3, f'{fmt(m)}%', ha='center', fontsize=12, fontweight='bold')
ax4.set_xticks(x); ax4.set_xticklabels(nomes, fontsize=13); ax4.set_ylim(0, 16)
ax4.set_ylabel('Chance de ficar entre os 10% mais respondidos do grupo (%)', fontsize=12)
ax4.legend(frameon=False, fontsize=12, loc='upper left')
ax4.set_title('Quem fala demais de moral tem menos chance de se espalhar\nPosts do mesmo tema e com a mesma carga moral, IC 95%', fontsize=14, pad=15)
plt.figtext(0.5, 0.005, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig('dia-168-grafico-04-a-chance-de-se-espalhar.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 4 salvo!")

# GRÁFICO 5 · Os 13 pontos ideais
fig5, ax5 = plt.subplots(figsize=(12, 8))
o = otimos.sort_values('otimo').reset_index(drop=True)
ys = np.arange(len(o) + 1)[::-1]
for y, (_, r) in zip(ys[:-1], o.iterrows()):
    ax5.errorbar(r.otimo, y, xerr=1.96 * r.ep, fmt='o', color=CORES_PLAT[r.plataforma], ms=9, capsize=4, lw=2)
ax5.errorbar(otimo_medio, ys[-1], xerr=1.96 * ep_medio, fmt='D', color='#222', ms=13, capsize=6, lw=3)
ax5.axvline(otimo_medio, color=CINZA, ls='--', lw=1.2)
ax5.set_yticks(ys); ax5.set_yticklabels([f'{r.tema} ({r.plataforma})' for _, r in o.iterrows()] + ['Média dos 13 temas'], fontsize=11)
ax5.text(otimo_medio + 1.96 * ep_medio + 0.01, ys[-1], f'{fmt(otimo_medio, 2)} [{fmt(ic_otimo[0], 2)}; {fmt(ic_otimo[1], 2)}]',
         va='center', fontsize=13, fontweight='bold')
ax5.set_xlim(0, 0.75)
ax5.set_xlabel('Densidade moral com o maior engajamento (IC 95%)', fontsize=12)
ax5.set_title('O ponto certo de moral muda de tema para tema\nPonto ideal de cada tema, calculado pelos autores', fontsize=14, pad=15)
plt.figtext(0.5, 0.005, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig('dia-168-grafico-05-os-pontos-ideais.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Gráfico 5 salvo!")

### 💡 O Insight

**Em 54 de cada 100 comparações, o post que fala demais de moral tem menos respostas do que um post equilibrado do mesmo tema.**

O intervalo de 95% é estreito, de 53,7 a 54,1, e fica todo acima do empate. A chance de o post ficar entre os 10% mais respondidos cai de 9,2% para 8,0%.

O efeito é pequeno em cada post e consistente nas três plataformas. No 8chan, o mais radical dos três, é maior, com 65 comparações perdidas em cada 100.

E falar demais é raro. Só 4% dos posts passam da densidade de 0,30, e menos de 1% passa de 0,50. A maior parte de quem fala de moral nas redes fala na medida.

A moral atrai. Cada palavra moral e emocional a mais se associa a 13% mais compartilhamentos, segundo a meta-análise de 2025. Mas o ponto em que ela para de ajudar muda de tema para tema, de 0,17 a 0,54.

A pergunta que fica:

*Quando você quer ser ouvido, você aponta o problema ou só julga?*

---

### ⚠️ Limitações do Modelo

- É uma análise de dados observacionais. Mostra associação, e não causa. Os dados não incluem o número de seguidores em todas as plataformas nem o histórico de quem publicou.
- A densidade moral é medida por um método automático, que compara as palavras do post a um dicionário moral. O corte de 0,30 segue o ponto médio encontrado pelos autores.
- "Respostas" quer dizer coisas diferentes em cada plataforma. No Twitter são retuítes, no Reddit são respostas diretas e no 8chan são as respostas somadas de um tema num dia, e não de um post.
- As comparações juntam posts do mesmo tema e do mesmo quinto de carga moral. Grupos com menos de 20 posts em algum dos lados ficaram de fora.
- O intervalo das comparações usa a fórmula de Hanley e McNeil, que trata os empates de forma aproximada. No Twitter, a maior parte dos posts tem zero retuítes, e por isso muitas comparações empatam.
- As discussões são todas em inglês, em plataformas de fora do Brasil. Não há dados do Brasil nem do Instagram.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*